In [ ]:
# Cell 1 — Install missing Colab packages (always first, no exceptions)
!pip install -r requirements-colab.txt -q

In [ ]:
# Cell 2 — Mount Google Drive
# Dataset + Section 3 metrics_cnn.json at: MyDrive/aerial_detection/
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Clone repo and add to sys.path
# Replace GITHUB_URL with your actual repo URL before running.
import subprocess, sys

GITHUB_URL = "https://github.com/YOUR_USERNAME/aerial-detection-production.git"
REPO_DIR   = "/content/aerial-detection-production"

result = subprocess.run(
    ["git", "clone", GITHUB_URL, REPO_DIR],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"sys.path[0]: {sys.path[0]}")

In [ ]:
# Cell 4 — Imports + Seeds
# stdlib
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

# third-party
import matplotlib.pyplot as plt
import numpy as np
import mlflow
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input

# internal (from cloned repo)
from training.data_loader import build_generators
from training.transfer_learning import (
    build_efficientnet_model,
    compile_stage1,
    unfreeze_top_n,
    compile_stage2,
    get_callbacks,
    build_comparison_report,
    FINE_TUNE_AT, EPOCHS_STAGE1, EPOCHS_STAGE2, LR_STAGE1, LR_STAGE2,
)
from training.evaluate import evaluate_model

# Seeds
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# File-based MLflow — no server required on Colab
# Model registry (Staging/Production stages) deferred to Section 8 (Docker + PostgreSQL)
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("aerial_efficientnet")

print(f"TensorFlow: {tf.__version__}")
print(f"MLflow:     {mlflow.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Cell 5 — Config (all constants here, never inline)
# EPOCHS_STAGE1=20, EPOCHS_STAGE2=10, LR_STAGE1=1e-3, LR_STAGE2=1e-5,
# FINE_TUNE_AT=20 — imported from transfer_learning module
IMAGE_SIZE   = (224, 224)
BATCH_SIZE   = 32
DROPOUT_RATE = 0.3   # Lower than CNN's 0.5 — EfficientNetB0 base already regularized

DRIVE_ROOT   = Path("/content/drive/MyDrive/aerial_detection")
DATASET_DIR  = DRIVE_ROOT / "classification_dataset"
EDA_STATS    = Path(REPO_DIR) / "data" / "eda" / "eda_stats.json"
MODELS_DIR   = Path(REPO_DIR) / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_MODELS = DRIVE_ROOT / "models"
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)

CHECKPOINT_S1 = str(MODELS_DIR / "effnet_stage1_best.h5")
CHECKPOINT_S2 = str(MODELS_DIR / "effnet_stage2_best.h5")  # production model
FINAL_MODEL   = str(MODELS_DIR / "efficientnet_final.h5")

# Section 3 metrics — needed for model comparison in Cell 14
CNN_METRICS = MODELS_DIR / "metrics_cnn.json"

assert DATASET_DIR.exists(), f"Dataset not found: {DATASET_DIR} — check Drive mount"
assert EDA_STATS.exists(),   f"eda_stats.json not found: {EDA_STATS} — run EDA notebook first"

print(f"Dataset:    {DATASET_DIR}")
print(f"EDA stats:  {EDA_STATS}")
print(f"CNN metrics exist: {CNN_METRICS.exists()}")
if not CNN_METRICS.exists():
    print("  WARNING: metrics_cnn.json not found — run 02_CustomCNN.ipynb first")
    print("  Cell 14 (comparison report) will be skipped.")

In [ ]:
# Cell 6 — Load eda_stats.json
with open(EDA_STATS) as f:
    eda_stats = json.load(f)

aug_cfg      = eda_stats["augmentation_config"]
class_weight = {int(k): v for k, v in eda_stats["class_weight"].items()}

print(f"class_weight:           {class_weight}")
print(f"embedding_preprocessing: {eda_stats['embedding_preprocessing']}")
print(f"normalization_decision:  {eda_stats['normalization_decision']}")
print()
print("Augmentation config (read from eda_stats.json — not redefined here):")
for k, v in aug_cfg.items():
    print(f"  {k}: {v}")

In [ ]:
# Cell 7 — Build data generators (EfficientNetB0 preprocessing)
#
# CRITICAL: pass preprocessing_fn=preprocess_input — NOT rescale=1/255.
# preprocess_input expects pixels in [0,255] and scales to [-1,1] internally.
# Passing rescale=1/255 first would give preprocess_input values in ~[0,1] → wrong range.
# The build_generators() function accepts preprocessing_fn to replace rescale entirely.

train_gen, valid_gen, test_gen, class_weight_loaded, class_indices = build_generators(
    dataset_dir=DATASET_DIR,
    eda_stats_path=EDA_STATS,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    preprocessing_fn=preprocess_input,
)

print(f"Class indices: {class_indices}")
print(f"Train: {train_gen.n}  Valid: {valid_gen.n}  Test: {test_gen.n}")

In [ ]:
# Cell 8 — Build EfficientNetB0 model + summary
# Stage 1: only the 4-layer head is trainable.
# Stage 2: unfreeze_top_n() adapts the top 20 conv layers of the base.

model = build_efficientnet_model(input_shape=(224, 224, 3), dropout_rate=DROPOUT_RATE)
model.summary()

base = model.get_layer("efficientnetb0")
print(f"\nEfficientNetB0 base layers: {len(base.layers)}")
print(f"  Base params:             {base.count_params():,}")
print(f"  Total model params:      {model.count_params():,}")
print(f"  Trainable params (head): {sum(tf.size(w).numpy() for w in model.trainable_weights):,}")
print(f"\nFINE_TUNE_AT={FINE_TUNE_AT} means top {FINE_TUNE_AT} of {len(base.layers)} base layers")
print(f"will be unfrozen in Stage 2 (all BatchNorm layers remain frozen).")

In [ ]:
# Cell 9 — Stage 1: train classification head only
# High LR (1e-3) is correct here — only the randomly-initialised Dense head trains.
# The frozen EfficientNetB0 base acts as a fixed feature extractor.

model = compile_stage1(model, learning_rate=LR_STAGE1)
callbacks_s1 = get_callbacks(CHECKPOINT_S1, patience=5, monitor="val_loss", mode="min")

train_start = time.time()

with mlflow.start_run(run_name="efficientnet_v1") as run:
    RUN_ID = run.info.run_id

    mlflow.log_params({
        "model":          "efficientnet",
        "base":           "EfficientNetB0",
        "image_size":     str(IMAGE_SIZE),
        "batch_size":     BATCH_SIZE,
        "epochs_stage1":  EPOCHS_STAGE1,
        "epochs_stage2":  EPOCHS_STAGE2,
        "lr_stage1":      LR_STAGE1,
        "lr_stage2":      LR_STAGE2,
        "fine_tune_at":   FINE_TUNE_AT,
        "dropout_rate":   DROPOUT_RATE,
        "class_weight_0": class_weight[0],
        "class_weight_1": class_weight[1],
        "preprocessing":  "efficientnet_preprocess_input",
        "optimizer":      "adam",
        "bn_frozen":      True,
    })
    mlflow.log_params({f"aug_{k}": v for k, v in aug_cfg.items()})

    print(f"Stage 1: training head only ({EPOCHS_STAGE1} epochs max, patience=5) ...")
    history_stage1 = model.fit(
        train_gen,
        validation_data=valid_gen,
        epochs=EPOCHS_STAGE1,
        callbacks=callbacks_s1,
        class_weight=class_weight,
        verbose=1,
    )

    EPOCHS_S1_RUN  = len(history_stage1.history["loss"])
    early_stopped_s1 = EPOCHS_S1_RUN < EPOCHS_STAGE1

    # Log Stage 1 metrics — steps 0 to EPOCHS_S1_RUN-1
    for i in range(EPOCHS_S1_RUN):
        mlflow.log_metric("loss",     history_stage1.history["loss"][i],     step=i)
        mlflow.log_metric("val_loss", history_stage1.history["val_loss"][i], step=i)
        if "accuracy" in history_stage1.history:
            mlflow.log_metric("accuracy",     history_stage1.history["accuracy"][i],     step=i)
            mlflow.log_metric("val_accuracy", history_stage1.history["val_accuracy"][i], step=i)

    mlflow.log_metrics({
        "stage1_epochs_run":    EPOCHS_S1_RUN,
        "stage1_early_stopped": int(early_stopped_s1),
    })

print(f"\nStage 1 done: {EPOCHS_S1_RUN}/{EPOCHS_STAGE1} epochs  early_stopped={early_stopped_s1}")
print(f"MLflow run ID: {RUN_ID}")

In [ ]:
# Cell 10 — Stage 2: fine-tune top 20 base layers at lr=1e-5
#
# unfreeze_top_n():
#   1. base.trainable = True
#   2. Freeze all but top FINE_TUNE_AT layers
#   3. Re-freeze ALL BatchNorm layers (preserve ImageNet running statistics)
#
# compile_stage2() MUST be called after unfreeze_top_n() — Keras picks up
# trainability changes only at compile time.
#
# Stage 2 metrics logged with step offset = EPOCHS_S1_RUN so MLflow shows
# a continuous 20+N epoch curve, not two overlapping 0-to-N curves.

unfreeze_top_n(model, n=FINE_TUNE_AT)
compile_stage2(model, learning_rate=LR_STAGE2)

trainable_s2 = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f"Stage 2 trainable params: {trainable_s2:,}")
print(f"  (Stage 1 head-only trainable params were: {sum(tf.size(w).numpy() for w in model.layers[-4].trainable_weights):,})")

callbacks_s2 = get_callbacks(CHECKPOINT_S2, patience=5, monitor="val_loss", mode="min")

with mlflow.start_run(run_id=RUN_ID):
    print(f"Stage 2: fine-tuning top {FINE_TUNE_AT} base layers ({EPOCHS_STAGE2} epochs max, patience=5) ...")
    history_stage2 = model.fit(
        train_gen,
        validation_data=valid_gen,
        epochs=EPOCHS_STAGE2,
        callbacks=callbacks_s2,
        class_weight=class_weight,
        verbose=1,
    )

    EPOCHS_S2_RUN    = len(history_stage2.history["loss"])
    early_stopped_s2 = EPOCHS_S2_RUN < EPOCHS_STAGE2
    best_epoch_s2    = int(np.argmin(history_stage2.history["val_loss"])) + 1
    final_lr_s2      = float(history_stage2.history.get("lr", [LR_STAGE2])[-1])
    total_train_time_s = time.time() - train_start

    # Stage 2 metrics logged with offset so steps don't overwrite Stage 1
    for i in range(EPOCHS_S2_RUN):
        step = EPOCHS_S1_RUN + i   # offset = actual Stage 1 epochs run
        mlflow.log_metric("loss",     history_stage2.history["loss"][i],     step=step)
        mlflow.log_metric("val_loss", history_stage2.history["val_loss"][i], step=step)
        if "accuracy" in history_stage2.history:
            mlflow.log_metric("accuracy",     history_stage2.history["accuracy"][i],     step=step)
            mlflow.log_metric("val_accuracy", history_stage2.history["val_accuracy"][i], step=step)

    mlflow.log_metrics({
        "stage2_epochs_run":    EPOCHS_S2_RUN,
        "stage2_early_stopped": int(early_stopped_s2),
        "total_epochs":         EPOCHS_S1_RUN + EPOCHS_S2_RUN,
        "train_time_min":       round(total_train_time_s / 60, 2),
    })

print(f"\nStage 2 done: {EPOCHS_S2_RUN}/{EPOCHS_STAGE2} epochs  early_stopped={early_stopped_s2}")
print(f"Best stage2 epoch: {best_epoch_s2}  Final LR: {final_lr_s2:.2e}")
print(f"Total training time: {total_train_time_s/60:.1f} min")

In [ ]:
# Cell 11 — Training Curves: Stage 1 + Stage 2 on continuous axes
# concat_histories() joins the two history dicts so the x-axis is 0..total_epochs.
# axvline at EPOCHS_S1_RUN marks where fine-tuning begins — interviewers ask about this.

def concat_histories(h1, h2):
    """Concatenate two Keras History objects into a single dict."""
    combined = {}
    for key in h1.history:
        if key in h2.history:
            combined[key] = h1.history[key] + h2.history[key]
    return combined

history_combined = concat_histories(history_stage1, history_stage2)
epochs_total     = len(history_combined["loss"])
divider_epoch    = EPOCHS_S1_RUN   # x-value where Stage 2 begins

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("EfficientNetB0 — Two-Stage Training History", fontsize=13)

x = range(1, epochs_total + 1)

for ax, key_train, key_val, title, ylabel in [
    (axes[0], "loss",     "val_loss",     "Loss",     "Loss"),
    (axes[1], "accuracy", "val_accuracy", "Accuracy", "Accuracy"),
]:
    if key_train not in history_combined:
        continue
    ax.plot(x, history_combined[key_train], label=f"Train {ylabel.lower()}",      color="steelblue")
    ax.plot(x, history_combined[key_val],   label=f"Validation {ylabel.lower()}", color="darkorange")
    ax.axvline(
        x=divider_epoch + 0.5, color="crimson", linestyle="--",
        label=f"Fine-tuning begins (epoch {divider_epoch + 1})",
    )
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{title} — Stage 1 → Stage 2")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
curves_path = MODELS_DIR / "training_curves_efficientnet.png"
plt.savefig(curves_path, dpi=100)
plt.show()
print(f"Saved: {curves_path}")

In [ ]:
# Cell 12 — Evaluate on test set
# EarlyStopping(restore_best_weights=True) ensures model holds Stage 2 best-epoch weights.
# evaluate_model() saves confusion_matrix_efficientnet.png + roc_curve_efficientnet.png.

eval_metrics = evaluate_model(
    model=model,
    test_generator=test_gen,
    model_name="efficientnet",
    output_dir=MODELS_DIR,
)

m_default = eval_metrics["metrics_at_default_threshold"]
m_opt     = eval_metrics["metrics_at_optimal_threshold"]
thresh    = eval_metrics["threshold_analysis"]

print("── Test metrics at threshold=0.5 (default) ──────────────────")
for k, v in m_default.items():
    print(f"  {k:<12}: {v}")

print(f"\n── Test metrics at Youden-optimal threshold ({thresh['optimal_threshold']:.3f}) ──")
for k, v in m_opt.items():
    print(f"  {k:<12}: {v}")

print(f"\n  TPR: {thresh['optimal_tpr']:.4f}  FPR: {thresh['optimal_fpr']:.4f}")
print(f"  Eval time: {eval_metrics['eval_time_s']:.2f}s")

In [ ]:
# Cell 13 — Save metrics_effnet.json + model .h5 to Drive
# training block is stage-aware — unlike metrics_cnn.json which has a flat training dict.
# The metrics block schema is IDENTICAL to metrics_cnn.json — build_comparison_report()
# reads the same keys from both files.

test_gen.reset()
test_loss, *_ = model.evaluate(test_gen, verbose=0)

metrics_effnet = {
    "model_name":   "efficientnet",
    "architecture": "EfficientNetB0 (frozen) + GAP + Dense(256) + sigmoid; fine-tuned top 20 layers",
    "training": {
        "stage1": {
            "epochs_run":    EPOCHS_S1_RUN,
            "epochs_max":    EPOCHS_STAGE1,
            "early_stopped": early_stopped_s1,
            "final_lr":      LR_STAGE1,
        },
        "stage2": {
            "epochs_run":    EPOCHS_S2_RUN,
            "epochs_max":    EPOCHS_STAGE2,
            "early_stopped": early_stopped_s2,
            "best_epoch":    best_epoch_s2,
            "final_lr":      final_lr_s2,
        },
        "fine_tune_at":  FINE_TUNE_AT,
        "total_epochs":  EPOCHS_S1_RUN + EPOCHS_S2_RUN,
    },
    "metrics": {
        "test_accuracy":  m_default["accuracy"],
        "test_precision": m_default["precision"],
        "test_recall":    m_default["recall"],
        "test_f1":        m_default["f1_score"],
        "test_auc":       m_default["auc"],
        "test_loss":      round(float(test_loss), 4),
    },
    "optimal_threshold": {
        "threshold":  eval_metrics["threshold_analysis"]["optimal_threshold"],
        "accuracy":   m_opt["accuracy"],
        "precision":  m_opt["precision"],
        "recall":     m_opt["recall"],
        "f1_score":   m_opt["f1_score"],
        "criterion":  "youden_j",
    },
    "artifacts": {
        "model_file":        FINAL_MODEL,
        "confusion_matrix":  str(MODELS_DIR / "confusion_matrix_efficientnet.png"),
        "roc_curve":         str(MODELS_DIR / "roc_curve_efficientnet.png"),
        "training_curves":   str(MODELS_DIR / "training_curves_efficientnet.png"),
        "mlflow_run_id":     RUN_ID,
        "mlflow_experiment": "aerial_efficientnet",
    },
    "training_time_minutes": round(total_train_time_s / 60, 2),
    "generated_at": datetime.now(timezone.utc).isoformat(),
}

metrics_path = MODELS_DIR / "metrics_effnet.json"
with open(metrics_path, "w") as f:
    json.dump(metrics_effnet, f, indent=2)

# Save full model (architecture + weights)
model.save(FINAL_MODEL)

# Copy to Drive
for src in [
    FINAL_MODEL,
    CHECKPOINT_S2,
    str(metrics_path),
    str(MODELS_DIR / "confusion_matrix_efficientnet.png"),
    str(MODELS_DIR / "roc_curve_efficientnet.png"),
    str(MODELS_DIR / "training_curves_efficientnet.png"),
]:
    if Path(src).exists():
        shutil.copy2(src, DRIVE_MODELS)
        print(f"Copied to Drive: {Path(src).name}")

print(f"\nmetrics_effnet.json:")
print(f"  test_accuracy:  {metrics_effnet['metrics']['test_accuracy']}")
print(f"  test_f1:        {metrics_effnet['metrics']['test_f1']}")
print(f"  test_auc:       {metrics_effnet['metrics']['test_auc']}")
print(f"  optimal_thr:    {metrics_effnet['optimal_threshold']['threshold']}")

In [ ]:
# Cell 14 — Model comparison report
# build_comparison_report() reads metrics_cnn.json + metrics_effnet.json,
# saves model_comparison.csv, and writes production_model.txt (one line: winner name).
# FastAPI Section 6 reads production_model.txt at startup to select which .h5 to load.

if not CNN_METRICS.exists():
    print("WARNING: metrics_cnn.json not found — skipping comparison.")
    print(f"  Expected at: {CNN_METRICS}")
    print("  Run 02_CustomCNN.ipynb first, then re-run this cell.")
    winner = None
else:
    winner = build_comparison_report(
        metrics_cnn_path=CNN_METRICS,
        metrics_effnet_path=metrics_path,
        output_dir=MODELS_DIR,
    )

    df = pd.read_csv(MODELS_DIR / "model_comparison.csv")
    print("── Model Comparison Report ───────────────────────────────")
    print(df.to_string(index=False))
    print(f"\n{'─'*55}")
    print(f"  WINNER: {winner.upper()}")
    print(f"  production_model.txt → '{winner}'")
    print(f"  FastAPI loads: models/{winner}_final.h5")
    print(f"{'─'*55}")

    # Copy comparison artefacts to Drive
    for name in ["model_comparison.csv", "production_model.txt"]:
        src = MODELS_DIR / name
        if src.exists():
            shutil.copy2(src, DRIVE_MODELS)
            print(f"Copied to Drive: {name}")

In [ ]:
# Cell 15 — MLflow: log final artifacts + zip mlruns to Drive
#
# Registry promotion (Staging → Production) is deferred to Section 8:
# file-based MLflow (file:./mlruns) does NOT support model registry stages.
# Section 8 (docker compose up) runs MLflow with PostgreSQL backend — registry works there.
# production_model.txt is the mechanism FastAPI uses to identify the winner on Colab.

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metrics({
        "test_accuracy":     metrics_effnet["metrics"]["test_accuracy"],
        "test_precision":    metrics_effnet["metrics"]["test_precision"],
        "test_recall":       metrics_effnet["metrics"]["test_recall"],
        "test_f1":           metrics_effnet["metrics"]["test_f1"],
        "test_auc":          metrics_effnet["metrics"]["test_auc"],
        "test_loss":         metrics_effnet["metrics"]["test_loss"],
        "optimal_threshold": metrics_effnet["optimal_threshold"]["threshold"],
        "optimal_f1":        metrics_effnet["optimal_threshold"]["f1_score"],
    })

    for artifact in [
        FINAL_MODEL,
        str(metrics_path),
        str(MODELS_DIR / "confusion_matrix_efficientnet.png"),
        str(MODELS_DIR / "roc_curve_efficientnet.png"),
        str(MODELS_DIR / "training_curves_efficientnet.png"),
    ]:
        if Path(artifact).exists():
            mlflow.log_artifact(artifact)

    if winner and (MODELS_DIR / "model_comparison.csv").exists():
        mlflow.log_artifact(str(MODELS_DIR / "model_comparison.csv"))

print(f"MLflow run {RUN_ID} finalised.")
print("Registry promotion → deferred to Section 8 (Docker MLflow + PostgreSQL).")

# Write production_model.txt to Drive (already written by build_comparison_report)
if winner:
    prod_txt = MODELS_DIR / "production_model.txt"
    print(f"\nproduction_model.txt → '{winner}'")
    print(f"FastAPI reads this at startup to decide which .h5 to load.")

# Zip mlruns to Drive
mlruns_zip = DRIVE_ROOT / "mlruns_efficientnet"
shutil.make_archive(str(mlruns_zip), "zip", root_dir=".", base_dir="mlruns")
print(f"Zipped mlruns → {mlruns_zip}.zip")

print("\n── Section 4 Complete ──────────────────────────────────────")
print(f"  model:      efficientnet_final.h5       (Drive: {DRIVE_MODELS})")
print(f"  metrics:    metrics_effnet.json          (Drive + MODELS_DIR)")
print(f"  comparison: model_comparison.csv         (Drive + MODELS_DIR)")
print(f"  winner:     production_model.txt → '{winner if winner else 'N/A'}'")
print(f"  mlflow:     run_id={RUN_ID}")
print("  registry:   deferred to Section 8")